# Week 6, Day 1: Eigenvectors: the directions a matrix only stretches

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w006/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])
for v in [np.array([1.0, 1.0]), np.array([1.0, -1.0]), np.array([1.0, 0.0])]:
    print(v, "->", A @ v)

### Exercise: Is it an eigenvector?

Write `eigenvalue_of(A, v)` that returns the eigenvalue if `v` is an eigenvector of `A`, and `None` otherwise. Estimate λ as $\frac{\mathbf{v} \cdot A\mathbf{v}}{\mathbf{v} \cdot \mathbf{v}}$ (the projection of $A\mathbf{v}$ onto $\mathbf{v}$, Week 4), then check that $A\mathbf{v}$ really equals $\lambda\mathbf{v}$ with `np.allclose`.

In [ ]:
def eigenvalue_of(A, v):
    Av = A @ v
    lam = ...
    ...

print(eigenvalue_of(A, np.array([1.0, 1.0])), eigenvalue_of(A, np.array([1.0, 0.0])))

In [ ]:
import numpy as np
def test_yes():
    "returns the eigenvalue for eigenvectors"
    A = np.array([[2.0, 1.0], [1.0, 2.0]])
    assert np.isclose(eigenvalue_of(A, np.array([1.0, 1.0])), 3)
    assert np.isclose(eigenvalue_of(A, np.array([-2.0, 2.0])), 1)

def test_no():
    "returns None when v is knocked off its line"
    A = np.array([[2.0, 1.0], [1.0, 2.0]])
    assert eigenvalue_of(A, np.array([1.0, 0.0])) is None

def test_negative_and_3d():
    "works for negative eigenvalues and larger matrices"
    B = np.diag([1.0, -2.0, 5.0])
    assert np.isclose(eigenvalue_of(B, np.array([0.0, 3.0, 0.0])), -2)
    assert eigenvalue_of(B, np.array([1.0, 1.0, 0.0])) is None

test_yes()
test_no()
test_negative_and_3d()
print("All checks passed")

In [ ]:
W = np.array([[0.5, 0.3, 0.0],
              [0.3, 0.4, 0.2],
              [0.0, 0.2, 0.6]])
vals, vecs = np.linalg.eig(W)
print("eigenvalues:", vals.round(3))
print("eigenvectors (columns):")
print(vecs.round(3))

### Exercise: Sort and verify

Sort the eigenvalues from largest to smallest, reordering the columns of `vecs` to match (use `np.argsort`). Then check all three eigen equations at once: $WV = V\Lambda$, where $\Lambda$ is the diagonal matrix of eigenvalues. Store the result of `np.allclose` in `all_good`.

In [ ]:
order = ...
vals, vecs = vals[order], vecs[:, order]
all_good = ...
print(vals.round(3)); print(all_good)

In [ ]:
import numpy as np
def test_sorted():
    "vals is sorted from largest to smallest"
    assert np.all(np.diff(vals) <= 1e-12)

def test_pairs():
    "each column of vecs is still the eigenvector of the matching eigenvalue"
    assert np.allclose(W @ vecs, vecs * vals)

def test_whole():
    "all_good is True: W V equals V times diag(vals) as a matrix equation"
    assert all_good is True

test_sorted()
test_pairs()
test_whole()
print("All checks passed")

### Exercise: The power method

Write `power_method(A, n_iter=200)`: start from a vector of ones, repeatedly multiply by `A` and divide by the length, then return the eigenvalue estimate $\mathbf{v} \cdot A\mathbf{v}$ (for a unit vector) and the vector. Compare its answer with the top eigenvector from the previous cell.

In [ ]:
def power_method(A, n_iter=200):
    v = np.ones(A.shape[0])
    for _ in range(n_iter):
        ...
    lam = ...
    return lam, v

lam, v = power_method(W)
print(round(lam, 4), v.round(3), "  from eig:", vals[0].round(4), vecs[:, 0].round(3))

In [ ]:
import numpy as np
def test_value():
    "finds the largest eigenvalue of W"
    lam, v = power_method(W)
    assert np.isclose(lam, np.max(np.linalg.eigvals(W)), atol=1e-6)

def test_vector():
    "returns a unit vector parallel to the top eigenvector"
    lam, v = power_method(W)
    assert np.isclose(np.linalg.norm(v), 1)
    assert np.allclose(W @ v, lam * v, atol=1e-6)

def test_other():
    "works on another symmetric matrix"
    M = np.array([[4.0, 1.0, 0.0], [1.0, 3.0, 1.0], [0.0, 1.0, 1.0]])
    lam, v = power_method(M)
    assert np.isclose(lam, np.max(np.linalg.eigvalsh(M)), atol=1e-6)

test_value()
test_vector()
test_other()
print("All checks passed")

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for trial in range(5):
    x = rng.uniform(0, 1, 3)
    xs = [x]
    for t in range(30):
        x = W @ x
        xs.append(x)
    xs = np.array(xs)
    ax[0].semilogy(np.linalg.norm(xs, axis=1), color="0.5")
    ax[1].plot(xs[:, 0] / np.linalg.norm(xs, axis=1), color="C0", alpha=0.7)
ax[0].semilogy(vals[0] ** np.arange(31) * 0.8, "k--", label=f"λ₁^t, λ₁ = {vals[0]:.2f}")
ax[0].set_xlabel("step t"); ax[0].set_ylabel("|x_t|"); ax[0].legend(frameon=False)
ax[1].axhline(abs(vecs[0, 0]), ls="--", color="k"); ax[1].set_xlabel("step t"); ax[1].set_ylabel("share of population 1")
plt.tight_layout(); plt.show()